# 03. Model Training

KSPHM-KIMM 2025 Bearing RUL Prediction - Model Training Notebook

This notebook covers:
1. CNN-LSTM model architecture
2. Data loading and preprocessing
3. Model training
4. Evaluation and inference

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.models import CNNLSTM, CNNLSTMConfig, BearingDataset, TestDataset, Trainer, TrainingConfig
from src.models.cnn_lstm import load_model
from src.utils import plot_training_history

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Model Architecture

In [ ]:
# Create model with default configuration
model = CNNLSTM(input_channels=15)

# Print model summary
print(model.summary())
print("\nModel Architecture:")
print(model)

In [ ]:
# Test forward pass with dummy data
batch_size = 4
time_steps = 256000
channels = 15

dummy_input = torch.randn(batch_size, time_steps, channels)
print(f"Input shape: {dummy_input.shape}")

model.eval()
with torch.no_grad():
    output = model(dummy_input)
print(f"Output shape: {output.shape}")

## 2. Load Pretrained Model

In [ ]:
# Load pretrained weights
WEIGHTS_PATH = "../weights/best_model_0.66.pth"

if os.path.exists(WEIGHTS_PATH):
    model = load_model(WEIGHTS_PATH, input_channels=15, device=device)
    print(f"Loaded pretrained model from: {WEIGHTS_PATH}")
else:
    print(f"Weights file not found: {WEIGHTS_PATH}")
    print("Will use randomly initialized model.")

## 3. Dataset Preparation

In [ ]:
# Configuration
DATA_DIR = "../data"
PREPROCESSED_DIR = f"{DATA_DIR}/preprocessed"

# Check if preprocessed data exists
if os.path.exists(PREPROCESSED_DIR):
    folders = os.listdir(PREPROCESSED_DIR)
    print(f"Found preprocessed folders: {folders}")
else:
    print(f"Preprocessed directory not found: {PREPROCESSED_DIR}")
    print("Please run the feature engineering notebook first.")

In [ ]:
# Example: Create test dataset for validation data
VALIDATION_DIR = f"{DATA_DIR}/Validation"

if os.path.exists(VALIDATION_DIR):
    # List validation folders
    val_folders = sorted(os.listdir(VALIDATION_DIR))
    print(f"Validation folders: {val_folders}")
    
    # Count CSV files
    total_files = 0
    for folder in val_folders:
        folder_path = os.path.join(VALIDATION_DIR, folder)
        if os.path.isdir(folder_path):
            csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
            total_files += len(csv_files)
            print(f"  {folder}: {len(csv_files)} files")
    print(f"Total validation files: {total_files}")

## 4. Training Configuration

In [ ]:
# Training configuration
config = TrainingConfig(
    batch_size=32,
    learning_rate=1e-3,
    epochs=100,
    early_stopping_patience=10,
    validation_split=0.2,
    use_log_transform=True,
    device="auto"
)

print("Training Configuration:")
print(f"  Batch size: {config.batch_size}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Max epochs: {config.epochs}")
print(f"  Early stopping patience: {config.early_stopping_patience}")
print(f"  Validation split: {config.validation_split}")
print(f"  Use log transform: {config.use_log_transform}")

## 5. Training Loop (Template)

In [ ]:
# This is a template for training
# Uncomment and modify as needed

'''
# Example RUL labels (replace with actual values)
rul_labels = {
    "file1.csv": 50000.0,
    "file2.csv": 45000.0,
    # ...
}

# Create dataset
dataset = BearingDataset(
    root_dir=PREPROCESSED_DIR,
    rul_values=rul_labels,
    normalize=True,
    log_transform_target=True
)

# Split data
from torch.utils.data import random_split
val_size = int(len(dataset) * 0.2)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Initialize model and trainer
model = CNNLSTM(input_channels=15)
trainer = Trainer(model, config)

# Train
history = trainer.train(
    train_loader, 
    val_loader, 
    save_path="../weights/trained_model.pth"
)

# Plot training history
plot_training_history(history)
plt.show()
'''

print("Training template ready. Uncomment and modify as needed.")

## 6. Inference

In [ ]:
def predict_rul(model, data_loader, use_log_transform=True):
    """
    Generate RUL predictions for test data.
    
    Args:
        model: Trained CNN-LSTM model
        data_loader: DataLoader with test data
        use_log_transform: Whether predictions are log-transformed
    
    Returns:
        List of (filename, predicted_rul) tuples
    """
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for x_test, fname in data_loader:
            x_test = x_test.to(device)
            pred = model(x_test)
            
            if use_log_transform:
                rul_sec = torch.expm1(pred).cpu().numpy().flatten()
            else:
                rul_sec = pred.cpu().numpy().flatten()
            
            for i, f in enumerate(fname):
                predictions.append((f, float(rul_sec[i])))
                print(f"{f} -> Predicted RUL: {rul_sec[i]:.2f} seconds")
    
    return predictions

In [ ]:
# Example inference (template)
'''
# Load test dataset
test_dataset = TestDataset(root_dir=VALIDATION_DIR, normalize=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Generate predictions
predictions = predict_rul(model, test_loader)

# Create submission DataFrame
submission = pd.DataFrame(predictions, columns=['Filename', 'RUL_Score (sec)'])
print("\nSubmission Preview:")
display(submission)

# Save submission
submission.to_excel("../results/submission.xlsx", index=False)
print("Submission saved!")
'''

print("Inference template ready. Uncomment and modify as needed.")

## 7. Model Evaluation Metrics

In [ ]:
def calculate_metrics(actual, predicted):
    """
    Calculate evaluation metrics for RUL prediction.
    
    Args:
        actual: Array of actual RUL values
        predicted: Array of predicted RUL values
    
    Returns:
        Dictionary of metrics
    """
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(actual - predicted))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    
    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    
    # R-squared
    ss_res = np.sum((actual - predicted) ** 2)
    ss_tot = np.sum((actual - np.mean(actual)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R-squared": r2
    }

# Example usage
# metrics = calculate_metrics(actual_rul, predicted_rul)
# print("Evaluation Metrics:")
# for name, value in metrics.items():
#     print(f"  {name}: {value:.4f}")

## 8. Summary

### Model Architecture
- **CNN Layer**: 64 filters, kernel size 5 - extracts local temporal features
- **MaxPool Layer**: kernel size 2 - reduces temporal dimension
- **LSTM Layer**: 64 hidden units - captures sequential patterns
- **FC Layers**: 64 -> 32 -> 1 - regression to RUL value

### Training Settings
- Loss: MSE (Mean Squared Error)
- Optimizer: Adam
- Target transformation: log1p (for scale normalization)
- Inverse transformation: expm1 (for prediction output)

### Best Model
- Validation score: 0.66
- Weights saved at: `weights/best_model_0.66.pth`